# All paths — raw speed and non-blocking numerical diagnostics

**Akseleu Omirbay.** Pinned atomic-ops revision `11616acd8a061768a38feb12c58e5e329952f5d5`.

## Запуск
Новая среда Colab **TPU v5e-1 → Выполнить все**. Пришлите `Atomic-gdn2-all-paths-speed-results-*.zip`.
Токены/Drive не нужны. Это полный набор 28 speed cases и serial diagnostic; компиляции и OLD E2E могут занять значительное время.

## Четыре явно различённые реализации
| Label | Реализация |
|---|---|
| NEW | Текущее `atomic_gdn2.pipeline.make_blr_trainable`, Pallas, bf16x3 solve/B3, zero damping |
| OLD | Affine `lax.associative_scan` из встроенного OLD; существующий recurrence remat, дополнительный block-remat только в E2E |
| JAX_REF | Чистая JAX chunked-WY реализация `atomic_ops.reference.gdn2_chunked_wy_reference` |
| PALLAS_V1 | Прежнее Pallas-ядро `atomic_ops.gdn2_pipeline.gdn2_pallas_forward_trainable`, прямой вызов без fallback |

Везде zero damping и chunk length 128. Конфигурации каждого пути сохраняются: NEW mb8, PALLAS_V1 mb16. Это разные реализации, не одинаковые kernel tuning parameters.

## Что измеряем
- Шесть kernel-форм: B4/D128, H=2/4/6, L=2048/4096. Gradient objective `sum(o**2)+sum(h_final**2)`, q/k/v/w/b/g; h0 захвачен.
- E2E: та же LM **34,873,952 параметра**, B8/L2048, исходный AdamW с clipping, synthetic tokens.
- **Три сида × три раунда. Каждый путь: 2 прогрева и 4 измерения в раунде.**
- Каждый вызов получает ОДНИ И ТЕ ЖЕ исходные входы/параметры/optimizer state. Выходы не становятся входами следующего вызова. Это фиксированный шаг, не эволюционирующая цепочка обучения.
- Полное ожидание результата. Компиляция, проверки конечности, CPU I/O и сохранение файлов вне таймеров.
- Один путь в одном отдельном процессе. Порядок путей ротируется между формами; внутри формы отдельные процессы не перемежаются по раундам. Одна TPU-сессия остаётся ограничением.
- Медианы раундов → медиана отношений на сид → mean ± sample SD по трём сидам.

## Численные расхождения НЕ блокируют таймер
Serial diagnostic и сравнение снимков с OLD/JAX_REF записываются отдельно при прежнем пороге 0.005. FAIL остаётся FAIL; никакой ошибки ядра не «исправляют» изменением допуска.
Kernel snapshot — шесть измеряемых градиентов; E2E snapshot — параметры, optimizer state и loss после одного шага, плюс разность параметров. Отдельные сырые градиенты всей LM в этом speed notebook не проверяются: прежние диагностические архивы сохраняются.

**OOM, исключения и nonfinite результаты не замалчиваются.** Ошибка одного пути сохраняется, остальные случаи продолжаются. Для незавершённого случая итоговая latency/speedup не объявляется.

## Что будет в результате
Абсолютные времена всех путей, ускорения относительно OLD и JAX_REF, каждый sample/warmup, numerical diagnostics, версии, исходники и SHA256. Большие снимки остаются в Colab, не в ZIP.

Этот протокол фиксированного состояния и равного числа вызовов отличается от прежнего evolving-state 4/2 E2E-протокола. Числа нужно подписывать именно этим протоколом, не смешивать между запусками. Raw speed не доказывает численную эквивалентность, одинаковые траектории обучения или качество модели.


In [ ]:
import sys,subprocess,pathlib,urllib.request,zipfile,hashlib,json,datetime
# Keep JAX out of the notebook process. Only one child process owns the TPU at a time.
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK=pathlib.Path('/content')/('atomic-all-paths-speed-'+RUN_ID);WORK.mkdir(exist_ok=True)
OUT=WORK/'results';OUT.mkdir()
# Colab's system Python may lack ensurepip; use virtualenv's bundled pip seed.
subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check','-q','virtualenv'])
subprocess.check_call([sys.executable,'-m','virtualenv',str(WORK/'venv')])
PY=str(WORK/'venv/bin/python')
PACKAGES=['jax[tpu]==0.11.2','jaxlib==0.11.2','libtpu==0.0.48.1','numpy==2.1.3',
 'scipy==1.16.3','flax==0.12.9','optax==0.2.8','ml_dtypes==0.6.0','opt_einsum==3.4.0']
subprocess.check_call([PY,'-m','pip','install','--disable-pip-version-check','-q',*PACKAGES])
REVISION='11616acd8a061768a38feb12c58e5e329952f5d5'
EXPECTED_SOURCE_HASHES={'atomic_gdn2/__init__.py': '7fcd8e1f64923cc5f4bc26aacda65a5ab797ee85c9e8a158d0ba9c8ee0f18142', 'atomic_gdn2/bwd/__init__.py': 'ebcdf69182f0a89a2227b1256cf3d9c011cb5ee04f486be9cf79a93c1e24a24a', 'atomic_gdn2/bwd/b4.py': '46f16bf7106a503c4b74536eeb4cb67cdf637f0eb738e15524cc5d39a5abd20e', 'atomic_gdn2/bwd/mega.py': '7af521b333bccdddb8e8e5e492631c7cfa786e9af77e4730c668d469b1dd0f17', 'atomic_gdn2/config.py': '2c84e4e2868a5ef233392a02b188045d87abeb16b7ac87bc1e18ed563c17e455', 'atomic_gdn2/domain.py': '34e88f51c9444b82b44b083a15f9dec46450c90b0beb9c49b992363cdd924ced', 'atomic_gdn2/fwd/__init__.py': 'f2b5315ff0c53b69b6fdc1ac35b39f5d3f0539ffa3a0cefd1195c8b49702889e', 'atomic_gdn2/fwd/cd_slim.py': 'd6dc8c1263ee9b640dc6e824dc411b46c9b771045ed5022e5a33ae2a8307b15b', 'atomic_gdn2/fwd/scores.py': '23d9db95633fd04bfae1ac162537bafe4ca5c180d74336da82eb948c69b7d728', 'atomic_gdn2/fwd/solve.py': '19b91ebbef82c2cd9d443d41b7a75ef6d7e82bbbd0dc925814c7a3d88413ede8', 'atomic_gdn2/layer.py': '6b18e1b925faa7664bc2fffcab4efab7c3b059b06f10aa7887518ed86566b112', 'atomic_gdn2/layout.py': '0df9dedbacdfab9c354ca6c95aba09e7df495d800388b5f2d9ddb5dc11f6769c', 'atomic_gdn2/pipeline.py': 'e7169d3f6d3ce2ed72fd14b25db1068aed2e8dce01c121a96ce35798e76297cf', 'atomic_gdn2/precision.py': '4bfd3485e20f9b1091a76f8bbcb2a98ef07fda3ba36151e18cda26350b140cc4', 'atomic_gdn2/reference/__init__.py': '7db881db3e8b52992ce3c9f7ab1568992aa649f3637c622ff76c9fec9f83106d', 'atomic_gdn2/reference/ref.py': '736cfb4a82844a3eb6cdd9347f0c9315f69881cc6b23b44ef4259042de447eec', 'atomic_gdn2/training/__init__.py': '7ccc6a2897ada0293f7d5c1c04da5568910148534a425e045559007fb04a632f', 'atomic_gdn2/training/canary.py': 'f634a11102602076ce12b17da3688fc7280d0f04dcb5848121b8ab086ed2a154', 'atomic_gdn2/training/data.py': '2622b8f048baa214559fc888e51d70146d6ed3509eade4fecd437a4328da8c79', 'atomic_gdn2/training/model.py': '2075623b09a2462ab24e62dc6acd84926ee505b4b797dfb1f459a827e1a3dfd7', 'atomic_gdn2/training/train.py': '00749c871fb6a9d8bbe0f436d456c2a65350de237f2f8d00de59b972e64fc3df', 'atomic_ops/__init__.py': 'cdafb768f2f4238253359e3b8041bf0106ae02575dfd1b4727ad90fd6c875896', 'atomic_ops/configs.py': '415907bf504a544ebf206890022a8b5518bad84862617acb627a1693d2112e61', 'atomic_ops/fallback.py': '0d7e5ec2403f0191956455c067a1acbb9a05f9aa35eb8f5f7ab0ae0a20f4a059', 'atomic_ops/gdn2_bwd.py': '9467914c55bc1d4d86c62af960e3827dc1e028a918df0ab5bd85bdf508774936', 'atomic_ops/gdn2_fwd.py': 'ba8926278a788063febae3d94574c665f22a3d9feb00bc46351f148f6e39a327', 'atomic_ops/gdn2_pipeline.py': '6a0c7e59c87c7615e296483145ade67caa1d1dd1ebb63c4e3c9b1212b9021fc5', 'atomic_ops/reference.py': 'ee7ecc97a2ef474e29219256646ee7d118045cc798798aaed6851fd0d246ad3d', 'atomic_ops/utils.py': '69c6bc145de9e2448a86b000c92b7724b6c830bd2f5e5b63b3658af254f876a2'}
archive=WORK/'source.zip'
urllib.request.urlretrieve('https://github.com/Akseleu-J/atomic-ops/archive/'+REVISION+'.zip',str(archive))
with zipfile.ZipFile(archive) as z:
 for n in z.namelist():
  assert not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts
 z.extractall(WORK/'source')
source=WORK/'source'/('atomic-ops-'+REVISION)
for n,h in EXPECTED_SOURCE_HASHES.items():assert hashlib.sha256((source/n).read_bytes()).hexdigest()==h,n
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True))
(OUT/'setup.json').write_text(json.dumps({'revision':REVISION,'source_hashes':EXPECTED_SOURCE_HASHES,'packages':PACKAGES,'run_id':RUN_ID},indent=2))
print('SETUP_OK',RUN_ID,'verified source files:',len(EXPECTED_SOURCE_HASHES),flush=True)
INTERMEDIATE=WORK/'intermediate';INTERMEDIATE.mkdir()
print('Fixed-state timing: four paths, six kernel shapes and E2E. Numerical deviations are non-blocking diagnostics.')
PROTOCOL={'id':'FIXED_STATE_ALL_PATHS_V1','paths':['OLD','JAX_REF','PALLAS_V1','NEW'],
 'seeds':[0,1,2],'rounds':3,'warmup_calls':2,'timed_calls_per_round':4,
 'fixed_state_replay':True,'full_tree_synchronization':True,'numerical_diagnostics_block_timing':False,
 'OLD_E2E_block_remat':True,'kernel_chunk_length':128,'NEW_wy_eps':0,
 'scope':'Raw latency; numerical disagreement stays visible. Not an automatic correctness or publication certificate.'}
(OUT/'protocol.json').write_text(json.dumps(PROTOCOL,indent=2))


In [ ]:
PAYLOAD={'candidate_common.py': '"""Shared corrected recurrence for the author\'s supplied historical benchmarks.\n\nRetains the affine associative-scan baseline and original checkpoint/custom-VJP\nstructure. Removes matrix normalization and gradient clipping; honors scale/h0/hf.\n"""\nimport dataclasses,hashlib,importlib.metadata,json,math,os,platform,statistics,sys,time\nfrom pathlib import Path\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT))\nfrom atomic_gdn2 import make_cfg,make_blr_trainable\nfrom atomic_ops.reference import gdn2_token_serial_reference, gdn2_chunked_wy_reference\nfrom atomic_gdn2.reference import forward_ref\nPREC=jax.lax.Precision.HIGHEST\nTOL=.005\nNAMES=(\'q\',\'k\',\'v\',\'w\',\'b\',\'g\',\'h0\')\n\ndef old_impl(q,k,v,w,b,g,h0,scale):\n B,L,H,D=q.shape;C=128;N=L//C\n assert L%C==0\n alpha=jnp.exp(g.astype(jnp.float32));ea=(b*k)*alpha;z=w*v\n def chunk(t):return jnp.moveaxis(t.reshape(B,N,C,H,D),1,0)\n def combine(s1,s2):\n  m1,c1=s1;m2,c2=s2\n  return jnp.matmul(m2,m1,precision=PREC),jnp.matmul(m2,c1,precision=PREC)+c2\n def step(carry,ci):\n  cM,cS=carry;kc,eac,zc,ac,qc=ci\n  M=jnp.eye(D,dtype=q.dtype)[None,None,None,:,:]*ac[...,None,:]-kc[..., :,None]*eac[...,None,:]\n  U=kc[..., :,None]*zc[...,None,:]\n  P,S=jax.lax.associative_scan(combine,(M,U),axis=1)\n  gM=jnp.einsum(\'bchmn,bhnp->bchmp\',P,cM,precision=PREC)\n  gS=jnp.einsum(\'bchmn,bhnp->bchmp\',P,cS,precision=PREC)+S\n  out=scale*jnp.einsum(\'bchij,bchi->bchj\',gS,qc,precision=PREC)\n  return (gM[:,-1],gS[:,-1]),out\n eye=jnp.broadcast_to(jnp.eye(D,dtype=q.dtype),(B,H,D,D))\n (_,hf),out=jax.lax.scan(jax.checkpoint(step),(eye,h0),tuple(map(chunk,(k,ea,z,alpha,q))))\n return jnp.moveaxis(out,0,1).reshape(B,L,H,D),hf\n\ndef old_factory(scale):\n def impl(*args):return old_impl(*args,scale)\n @jax.custom_vjp\n def fn(*args):return impl(*args)\n def fwd(*args):\n  out,pullback=jax.vjp(jax.checkpoint(impl),*args)\n  return out,pullback\n def bwd(pullback,cotangent):return pullback(cotangent)\n fn.defvjp(fwd,bwd)\n return fn\n\ndef paths(D=128,profile=\'strict\',cpu=False):\n scale=1/math.sqrt(D);old=old_factory(scale)\n def serial(q,k,v,w,b,g,h0):return gdn2_token_serial_reference(q,k,v,g,b,w,scale,h0)\n def reference(q,k,v,w,b,g,h0):return gdn2_chunked_wy_reference(q,k,v,g,b,w,scale,128,h0,wy_eps=0.)\n result={\'OLD\':old,\'SERIAL\':serial,\'JAX_REF\':reference};cfg=None\n if not cpu:\n  strict=profile==\'strict\'\n  config=make_cfg(bt=128,bc=64,score_bs=128,bs2=64,mb=8,diag=\'btl\',\n   wy_eps=0. if strict or profile==\'no_damping\' else .001,dot_mode=\'highest\',solve_dot_mode=\'highest\' if strict else \'bf16x3\',\n   b3_dot_mode=\'highest\' if strict else \'bf16x3\',interpret=False)\n  kernel=make_blr_trainable(config,scale)\n  def new(q,k,v,w,b,g,h0):\n   sw=lambda x:jnp.swapaxes(x,1,2)\n   o,h=kernel(*(sw(x) for x in (q,k,v,w,b,g)),h0)\n   return sw(o),h\n  # Fail before timing if the candidate closure does not originate in atomic_gdn2.\n  import inspect\n  assert kernel.__module__==\'atomic_gdn2.pipeline\',kernel.__module__\n  assert Path(inspect.getsourcefile(kernel)).resolve()==(ROOT/\'atomic_gdn2/pipeline.py\').resolve()\n  assert config.interpret is False and config.wy_eps==0.0\n  assert result[\'OLD\'] is old and result[\'JAX_REF\'] is reference and old is not reference\n  result[\'NEW\']=new;cfg=dataclasses.asdict(config)\n return result,cfg\n\ndef inputs(seed,shape=(4,2048,6,128)):\n B,L,H,D=shape\n keys=jax.random.split(jax.random.PRNGKey(seed),7)\n q=jax.random.normal(keys[0],shape)*.3;k=jax.random.normal(keys[1],shape)*.3\n q=q/(jnp.linalg.norm(q,axis=-1,keepdims=True)+1e-6)\n k=k/(jnp.linalg.norm(k,axis=-1,keepdims=True)+1e-6)\n v=jax.random.normal(keys[2],shape)*.3;g=-jnp.abs(jax.random.normal(keys[3],shape))*.05\n b=jax.random.uniform(keys[4],shape,minval=.2,maxval=1.)\n w=jax.random.uniform(keys[5],shape,minval=.2,maxval=1.)\n h0=jax.random.normal(keys[6],(B,H,D,D))*.1\n args=tuple(jax.device_put(t) for t in (q,k,v,w,b,g,h0));jax.block_until_ready(args)\n digest=hashlib.sha256(b\'\'.join(np.asarray(x).tobytes() for x in args)).hexdigest()\n return args,digest\n\ndef loss_for(fn):\n def loss(*args):\n  o,h=fn(*args);return jnp.sum(o.astype(jnp.float32)**2)+jnp.sum(h.astype(jnp.float32)**2)\n return loss\n\ndef check(got,ref,tolerance=TOL):\n assert jax.tree.structure(got)==jax.tree.structure(ref),\'Different result trees\'\n values=[]\n paths=[jax.tree_util.keystr(p) for p,_ in jax.tree_util.tree_flatten_with_path(ref)[0]]\n for path,a,b in zip(paths,jax.tree.leaves(got),jax.tree.leaves(ref)):\n  a=np.asarray(a,np.float64);b=np.asarray(b,np.float64)\n  assert a.shape==b.shape,(a.shape,b.shape)\n  finite=bool(np.isfinite(a).all() and np.isfinite(b).all())\n  rel=float(np.linalg.norm((a-b).reshape(-1))/(np.linalg.norm(b.reshape(-1))+1e-12)) if finite else None\n  values.append(dict(path=path,shape=list(a.shape),finite=finite,rel_l2=rel,\n   max_abs=float(np.max(np.abs(a-b))) if finite else None,reference_norm=float(np.linalg.norm(b.reshape(-1))) if finite else None))\n return dict(passed=bool(values) and all(v[\'finite\'] and v[\'rel_l2\']<=tolerance for v in values),tensors=values)\n\ndef compile_fn(fn,args):\n start=time.perf_counter();exe=jax.jit(fn).lower(*args).compile()\n return exe,time.perf_counter()-start\n\ndef timed(exe,args,repeats,warmup):\n warm=[]\n for _ in range(warmup):\n  start=time.perf_counter();jax.block_until_ready(exe(*args));warm.append((time.perf_counter()-start)*1000)\n samples=[]\n for _ in range(repeats):\n  start=time.perf_counter();jax.block_until_ready(exe(*args));samples.append((time.perf_counter()-start)*1000)\n return summary(samples)|dict(warmup_samples_ms=warm,repeats=repeats,warmup=warmup)\n\ndef summary(samples):\n return dict(samples_ms=samples,median_ms=statistics.median(samples),mean_ms=statistics.mean(samples),\n  population_sd_ms=statistics.pstdev(samples),sample_sd_ms=statistics.stdev(samples) if len(samples)>1 else 0.)\n\ndef environment():\n versions={}\n for name in (\'jax\',\'jaxlib\',\'libtpu\',\'numpy\',\'flax\',\'optax\'):\n  try:versions[name]=importlib.metadata.version(name)\n  except importlib.metadata.PackageNotFoundError:versions[name]=None\n return dict(python=sys.version,platform=platform.platform(),backend=jax.default_backend(),\n  device_kinds=[x.device_kind for x in jax.devices()],devices=[str(x) for x in jax.devices()],\n  implementation_identity=dict(candidate_module=\'atomic_gdn2.pipeline\',candidate_source=str((ROOT/\'atomic_gdn2/pipeline.py\').resolve()),candidate_source_sha256=hashlib.sha256((ROOT/\'atomic_gdn2/pipeline.py\').read_bytes()).hexdigest(),baseline=\'candidate_common.old_impl: affine associative_scan\',pure_JAX_reference=\'atomic_ops.reference.gdn2_chunked_wy_reference: validation only\'),\n  packages=versions,matmul_precision=str(jax.config.jax_default_matmul_precision),\n  flags={n:os.environ.get(n) for n in (\'XLA_FLAGS\',\'JAX_PLATFORMS\')})\n\ndef save(report,dest):\n report[\'executed_benchmark_sources\']={f.name:hashlib.sha256(f.read_bytes()).hexdigest()\n  for f in Path(__file__).parent.glob(\'*.py\')}\n report[\'source_revision\']=\'11616acd8a061768a38feb12c58e5e329952f5d5\'\n report[\'kernel_source_sha256\']={str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest() for f in (ROOT/\'atomic_gdn2\').rglob(\'*.py\')}\n Path(dest).write_text(json.dumps(report,indent=2,allow_nan=False),encoding=\'utf-8\')\n\ndef require_tpu():\n assert jax.default_backend()==\'tpu\' and jax.local_device_count()==1,\'Speed measurements require one real TPU\'\n', 'candidate_e2e.py': '"""Author\'s original eight-layer LM/AdamW, matched exact recurrence and full-step waits."""\nimport argparse,dataclasses,hashlib,json,statistics,time,traceback\nimport candidate_common as c\nimport optax\nfrom corrected_original_lm import LMConfig,LM,SUPPLIED_MODEL_SHA256\n\ndef adapter(fn,head_dim):\n def run(q,k,v,w,b,g):\n  h0=c.jnp.zeros((q.shape[0],q.shape[2],head_dim,head_dim),c.jnp.float32)\n  return fn(q,k,v,w,b,g,h0)\n return run\n\ndef initialize(cfg,seed,tokens):\n def dummy(q,k,v,w,b,g):return q,c.jnp.zeros((q.shape[0],q.shape[2],q.shape[3],q.shape[3]),c.jnp.float32)\n model=LM(cfg=cfg,kernel_fn=dummy)\n params=model.init(c.jax.random.PRNGKey(seed),tokens)[\'params\']\n return c.jax.tree.map(c.jax.device_put,params)\n\ndef make_functions(cfg,fn):\n model=LM(cfg=cfg,kernel_fn=adapter(fn,cfg.d_head))\n tx=optax.chain(optax.clip_by_global_norm(1.),optax.adamw(1e-3,weight_decay=.01))\n def loss(params,ids):\n  logits=model.apply({\'params\':params},ids)\n  return optax.softmax_cross_entropy_with_integer_labels(logits[:,:-1],ids[:,1:]).mean()\n def step(params,state,ids):\n  val,grads=c.jax.value_and_grad(loss)(params,ids)\n  updates,new_state=tx.update(grads,state,params)\n  return optax.apply_updates(params,updates),new_state,val\n return loss,step,tx\n\ndef timed_step(exe,params,state,tokens,repeats,warmup=2):\n # Exactly the supplied evolving-state loop, with full result-tree synchronization fixed.\n p,s=params,state;warm=[]\n for _ in range(warmup):\n  start=time.perf_counter();out=exe(p,s,tokens);c.jax.block_until_ready(out)\n  warm.append((time.perf_counter()-start)*1000);p,s,_=out\n samples=[]\n for _ in range(repeats):\n  start=time.perf_counter();out=exe(p,s,tokens);c.jax.block_until_ready(out)\n  samples.append((time.perf_counter()-start)*1000);p,s,_=out\n finite=all(bool(c.np.isfinite(c.np.asarray(x)).all()) for x in c.jax.tree.leaves(out))\n assert finite,\'Nonfinite result during evolving-state timing\'\n return c.summary(samples)|dict(warmup_samples_ms=warm,repeats=repeats,warmup=warmup,\n  input_positions_per_second=tokens.size/statistics.median(samples)*1000,\n  predicted_tokens_per_second=tokens.shape[0]*(tokens.shape[1]-1)/statistics.median(samples)*1000,\n  state_evolves=True,full_tree_synchronized=True,final_results_finite=finite,final_loss=float(out[2]))\n\ndef run(a):\n c.jax.config.update(\'jax_default_matmul_precision\',\'highest\');c.require_tpu()\n cfg=LMConfig();p,kernel_cfg=c.paths(cfg.d_head,a.profile)\n report=dict(status=\'running\',kind=\'old_comparison_e2e\',profile=a.profile,config=dataclasses.asdict(cfg),\n  kernel_config=kernel_cfg,environment=c.environment(),supplied_model_sha256=SUPPLIED_MODEL_SHA256,\n  batch=8,length=2048,microbatch=None,tolerance_rel_l2=c.TOL,\n  optimizer=\'clip_by_global_norm(1.0), AdamW(lr=1e-3, weight_decay=.01), default optax moments/eps\',\n  protocol=\'Original order NEW then OLD; two evolving-state warmups; NEW4/OLD2 samples per round; median/population SD; full result synchronization\',\n  note=\'Same historical architecture; actual parameter count reported from model.init; counted from the initialized parameter tree. Synthetic uniform tokens, no corpus I/O.\',seeds=[])\n functions={name:make_functions(cfg,p[name]) for name in (\'NEW\',\'OLD\')}\n exes={};gradfs={};shared_compile_s={}\n for seed in a.seeds:\n  host=c.np.random.default_rng(seed).integers(0,cfg.vocab_size,size=(8,2048))\n  tokens=c.jax.device_put(c.jnp.asarray(host,c.jnp.int32));params=initialize(cfg,seed,tokens)\n  count=sum(x.size for x in c.jax.tree.leaves(params));report[\'params\']=count\n  assert count==34873952,(\'Unexpected historical architecture parameter count\',count)\n  states={};compile_s={}\n  for name in (\'NEW\',\'OLD\'):\n   loss,step,tx=functions[name];states[name]=tx.init(params)\n   if name not in exes:\n    print(\'CORRECTED_E2E_COMPILE\',seed,name,\'gradient\',flush=True)\n    gradfs[name],cg=c.compile_fn(c.jax.value_and_grad(loss),(params,tokens))\n    print(\'CORRECTED_E2E_COMPILE\',seed,name,\'AdamW_step\',flush=True)\n    exes[name],cs=c.compile_fn(step,(params,states[name],tokens))\n    compile_s[name]=dict(gradient_seconds=cg,step_seconds=cs)\n    shared_compile_s[name]=compile_s[name]\n  c.jax.block_until_ready((params,tokens,states))\n  checks=dict(raw_parameter_gradients=c.check(gradfs[\'NEW\'](params,tokens),gradfs[\'OLD\'](params,tokens)),\n   full_optimizer_update=c.check(exes[\'NEW\'](params,states[\'NEW\'],tokens),exes[\'OLD\'](params,states[\'OLD\'],tokens)))\n  np_,ns_,nl_=exes[\'NEW\'](params,states[\'NEW\'],tokens)\n  rp_,rs_,rl_=exes[\'OLD\'](params,states[\'OLD\'],tokens)\n  checks[\'parameter_updates\']=c.check(c.jax.tree.map(lambda p,x:p-x,np_,params),c.jax.tree.map(lambda p,x:p-x,rp_,params))\n  row=dict(seed=seed,input_sha256=hashlib.sha256(host.tobytes()).hexdigest(),checks=checks,compile_seconds=compile_s,executables_reused_across_seeds=True,\n   passed=all(x[\'passed\'] for x in checks.values()),rounds=[])\n  if row[\'passed\']:\n   for rnd in range(a.rounds):\n    times={name:timed_step(exes[name],params,states[name],tokens,4 if name==\'NEW\' else 2) for name in (\'NEW\',\'OLD\')}\n    row[\'rounds\'].append(dict(round=rnd,timing=times,speedup=times[\'OLD\'][\'median_ms\']/times[\'NEW\'][\'median_ms\']))\n   row[\'speedup\']=statistics.median(r[\'speedup\'] for r in row[\'rounds\'])\n  report[\'seeds\'].append(row);c.save(report,a.output)\n  print(\'CORRECTED_E2E_SEED\',seed,row[\'passed\'],row.get(\'speedup\'),flush=True)\n report[\'status\']=\'passed\' if all(r[\'passed\'] for r in report[\'seeds\']) else \'gate_failed\'\n report[\'shared_compile_seconds\']=shared_compile_s\n if report[\'status\']==\'passed\':\n  ratios=[r[\'speedup\'] for r in report[\'seeds\']]\n  report[\'speedup\']=dict(per_seed=ratios,mean=statistics.mean(ratios),sample_sd=statistics.stdev(ratios) if len(ratios)>1 else 0.,\n   historical_single_trial=report[\'seeds\'][0][\'rounds\'][0][\'speedup\'],summary_method=\'median of round ratios per seed; mean and sample SD across seeds\')\n return report\n\ndef cpu_check():\n c.jax.config.update(\'jax_default_matmul_precision\',\'highest\')\n assert c.jax.default_backend()==\'cpu\'\n # Actual full-model parameter count (dummy recurrence during init).\n cfg=LMConfig();tokens=c.jnp.zeros((1,128),c.jnp.int32);params=initialize(cfg,0,tokens)\n count=sum(x.size for x in c.jax.tree.leaves(params));assert count==34873952\n del params\n small=LMConfig(d_model=8,n_heads=2,d_head=4,n_layers=2,mlp_mult=2)\n p,_=c.paths(4,cpu=True);args,_=c.inputs(0,(1,256,2,4))\n oracle=c.jax.jit(c.jax.value_and_grad(c.loss_for(p[\'SERIAL\']),argnums=tuple(range(7))))(*args)\n gate=c.check(c.jax.jit(c.jax.value_and_grad(c.loss_for(p[\'OLD\']),argnums=tuple(range(7))))(*args),oracle)\n assert gate[\'passed\'],gate\n actual_dim,_=c.paths(128,cpu=True);dim_args,_=c.inputs(1,(1,128,1,128))\n dim_oracle=c.jax.jit(c.jax.value_and_grad(c.loss_for(actual_dim[\'SERIAL\']),argnums=tuple(range(7))))(*dim_args)\n dim_gate=c.check(c.jax.jit(c.jax.value_and_grad(c.loss_for(actual_dim[\'OLD\']),argnums=tuple(range(7))))(*dim_args),dim_oracle)\n assert dim_gate[\'passed\'],dim_gate\n ids=c.jnp.asarray(c.np.random.default_rng(0).integers(0,256,(1,128)),c.jnp.int32)\n params=initialize(small,0,ids);fns={k:make_functions(small,p[k]) for k in (\'OLD\',\'SERIAL\')}\n results={};grads={}\n for name,(loss,step,tx) in fns.items():\n  grads[name]=c.jax.jit(c.jax.value_and_grad(loss))(params,ids)\n  results[name]=c.jax.jit(step)(params,tx.init(params),ids)\n checks=dict(kernel_seven_gradients_two_chunks=gate,kernel_seven_gradients_D128=dim_gate,lm_raw_gradients=c.check(grads[\'OLD\'],grads[\'SERIAL\']),\n  lm_AdamW_update=c.check(results[\'OLD\'],results[\'SERIAL\']))\n assert all(x[\'passed\'] for x in checks.values()),checks\n print(json.dumps(dict(status=\'cpu_checks_passed\',environment=c.environment(),full_historical_params=count,small_cfg=dataclasses.asdict(small),\n  checks=checks,note=\'CPU mathematical checks only. No Pallas/TPU performance measured.\'),indent=2),flush=True)\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--profile\',choices=(\'strict\',\'production\',\'no_damping\'),default=\'production\')\n p.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[0,1,2]);p.add_argument(\'--rounds\',type=int,default=3)\n p.add_argument(\'--output\',default=\'corrected-original-e2e.json\');p.add_argument(\'--cpu-check\',action=\'store_true\')\n a=p.parse_args();assert a.rounds>=1 and len(set(a.seeds))==len(a.seeds)\n if a.cpu_check:cpu_check();return 0\n try:r=run(a)\n except Exception as e:r=dict(status=\'error\',error=type(e).__name__,detail=str(e),traceback=traceback.format_exc(),environment=c.environment())\n c.save(r,a.output);print(\'FINAL_CORRECTED_E2E\',r[\'status\'],r.get(\'speedup\'),flush=True)\n return 0 if r[\'status\']==\'passed\' else 1\n\nif __name__==\'__main__\':raise SystemExit(main())\n', 'corrected_original_lm.py': '"""Architecture copied from the author-supplied historical E2E snippet.\nOnly formatting changes from AST unparsing; no architectural substitutions.\n"""\nimport dataclasses as dc,math\nimport jax\nimport jax.numpy as jnp\nfrom flax import linen as nn\nSUPPLIED_MODEL_SHA256=\'3c1e26ba720240b6fdde89b1bb44e9248f81a7890ec782a3b544aad185fd5992\'\n\n@dc.dataclass(frozen=True)\nclass LMConfig:\n    vocab_size: int = 256\n    d_model: int = 512\n    n_heads: int = 6\n    d_head: int = 128\n    n_layers: int = 8\n    mlp_mult: int = 2\n    d_conv: int = 4\n    g_max: float = 160.0 / 128\n    decay_init: float = 0.05\n\n    @property\n    def proj_dim(self):\n        return self.n_heads * self.d_head\n\nclass GDN2Block(nn.Module):\n    cfg: LMConfig\n    kernel_fn: object\n\n    @nn.compact\n    def __call__(self, x):\n        cfg = self.cfg\n        B, L, _ = x.shape\n        H, D = (cfg.n_heads, cfg.d_head)\n        proj_dim = H * D\n        eps = 1e-06\n\n        def sc(name, u):\n            dim = u.shape[-1]\n            w_ = self.param(f\'{name}_conv_w\', nn.initializers.normal(0.02), (dim, cfg.d_conv))\n            bc = self.param(f\'{name}_conv_b\', nn.initializers.zeros, (dim,))\n            rhs = w_.T[:, None, :].astype(u.dtype)\n            return jax.lax.conv_general_dilated(u, rhs, window_strides=(1,), padding=[(cfg.d_conv - 1, 0)], feature_group_count=dim, dimension_numbers=(\'NHC\', \'HIO\', \'NHC\')) + bc[None, None, :].astype(u.dtype)\n        q = jax.nn.silu(sc(\'q\', nn.Dense(proj_dim, use_bias=False, name=\'q_proj\')(x))).reshape(B, L, H, D)\n        k = jax.nn.silu(sc(\'k\', nn.Dense(proj_dim, use_bias=False, name=\'k_proj\')(x))).reshape(B, L, H, D)\n        v = jax.nn.silu(sc(\'v\', nn.Dense(proj_dim, use_bias=False, name=\'v_proj\')(x))).reshape(B, L, H, D)\n        v = jnp.clip(v, -50.0, 50.0)\n        l2n = lambda t: t * jax.lax.rsqrt(jnp.sum(t * t, -1, keepdims=True) + eps)\n        q = l2n(q)\n        k = l2n(k)\n        beta = jax.nn.sigmoid(nn.Dense(proj_dim, name=\'erase_gate\')(x)).reshape(B, L, H, D)\n        wg = jax.nn.sigmoid(nn.Dense(proj_dim, name=\'write_gate\')(x)).reshape(B, L, H, D)\n        bias0 = math.log(math.expm1(cfg.decay_init))\n        da = self.param(\'decay_a\', lambda k_, s: jnp.full(s, bias0, jnp.float32), (H,)).astype(jnp.float32)\n        df = nn.Dense(H, use_bias=True, name=\'decay_proj\')(x).astype(jnp.float32)\n        ldh = -jnp.exp(jnp.clip(da, -20, 20))[None, None, :] * jax.nn.softplus(df)\n        ld = jnp.broadcast_to(ldh[..., None], (B, L, H, D))\n        ld = jnp.nan_to_num(ld, nan=0.0, posinf=0.0, neginf=-20.0)\n        g = -jnp.minimum(jnp.abs(ld), cfg.g_max)\n        o, _ = self.kernel_fn(q, k, v, wg, beta, g)\n        o = o.reshape(B, L, proj_dim)\n        og = nn.Dense(proj_dim, use_bias=False, name=\'out_gate\')(x)\n        o = nn.RMSNorm(epsilon=1e-06, name=\'mixer_out_norm\')(o)\n        x = x + nn.Dense(cfg.d_model, use_bias=False, name=\'out_proj\')(o * jax.nn.silu(og))\n        h2 = nn.RMSNorm(epsilon=1e-06, name=\'mlp_norm\')(x)\n        d = x.shape[-1]\n        m = nn.Dense(d, use_bias=False, name=\'down_proj\')(jax.nn.silu(nn.Dense(cfg.mlp_mult * d, use_bias=False, name=\'gate_proj\')(h2)) * nn.Dense(cfg.mlp_mult * d, use_bias=False, name=\'up_proj\')(h2))\n        return x + m\n\nclass LM(nn.Module):\n    cfg: LMConfig\n    kernel_fn: object\n\n    @nn.compact\n    def __call__(self, ids):\n        emb = nn.Embed(num_embeddings=self.cfg.vocab_size, features=self.cfg.d_model, name=\'embed\')\n        x = emb(ids)\n        for i in range(self.cfg.n_layers):\n            x = GDN2Block(cfg=self.cfg, kernel_fn=self.kernel_fn, name=f\'block_{i}\')(x)\n        return emb.attend(nn.RMSNorm(epsilon=1e-06, name=\'final_norm\')(x))\n', 'isolated_e2e.py': '"""One path/one executable per process. Full-size LM, host-persisted gates."""\nimport argparse,dataclasses,gc,hashlib,json,statistics,time,traceback\nfrom pathlib import Path\nimport candidate_common as c\nfrom candidate_e2e import initialize,timed_step,adapter\nfrom corrected_original_lm import LMConfig,LM,GDN2Block\nimport optax\nfrom flax import linen as nn\n\nclass RematLM(nn.Module):\n cfg: LMConfig\n kernel_fn: object\n @nn.compact\n def __call__(self,ids):\n  emb=nn.Embed(num_embeddings=self.cfg.vocab_size,features=self.cfg.d_model,name=\'embed\')\n  x=emb(ids)\n  # Same module names and shapes as the supplied LM. Recompute blocks during AD.\n  Block=nn.remat(GDN2Block,prevent_cse=True)\n  for i in range(self.cfg.n_layers):x=Block(cfg=self.cfg,kernel_fn=self.kernel_fn,name=f\'block_{i}\')(x)\n  return emb.attend(nn.RMSNorm(epsilon=1e-6,name=\'final_norm\')(x))\n\ndef functions(cfg,fn,remat):\n model=(RematLM if remat else LM)(cfg=cfg,kernel_fn=adapter(fn,cfg.d_head))\n tx=optax.chain(optax.clip_by_global_norm(1.),optax.adamw(1e-3,weight_decay=.01))\n def loss(params,ids):\n  logits=model.apply({\'params\':params},ids)\n  return optax.softmax_cross_entropy_with_integer_labels(logits[:,:-1],ids[:,1:]).mean()\n def step(params,state,ids):\n  val,grads=c.jax.value_and_grad(loss)(params,ids)\n  updates,state=tx.update(grads,state,params)\n  return optax.apply_updates(params,updates),state,val\n return loss,step,tx\n\ndef persist(tree,dest):\n leaves=c.jax.tree_util.tree_flatten_with_path(tree)[0]\n arrays={f\'t{i:04d}\':c.np.asarray(v) for i,(_,v) in enumerate(leaves)}\n metadata=[{\'key\':f\'t{i:04d}\',\'path\':c.jax.tree_util.keystr(p),\'shape\':list(v.shape),\'dtype\':str(v.dtype)} for i,(p,v) in enumerate(leaves)]\n c.np.savez(dest,**arrays)\n meta=dest.with_suffix(\'.meta.json\');meta.write_text(json.dumps(metadata,indent=2))\n with dest.open(\'rb\') as f:digest=hashlib.file_digest(f,\'sha256\').hexdigest()\n return dict(file=dest.name,bytes=dest.stat().st_size,sha256=digest,metadata=metadata)\n\ndef fingerprint(tree):\n h=hashlib.sha256()\n for path,v in c.jax.tree_util.tree_flatten_with_path(tree)[0]:\n  a=c.np.asarray(v);h.update(c.jax.tree_util.keystr(path).encode());h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes())\n return h.hexdigest()\n\ndef run(a,report):\n c.jax.config.update(\'jax_default_matmul_precision\',\'highest\');c.require_tpu()\n cfg=LMConfig();paths,kc=c.paths(profile=\'no_damping\');use_remat=a.path==\'OLD\'\n loss,step,tx=functions(cfg,paths[a.path],use_remat)\n report.update(config=dataclasses.asdict(cfg),kernel_config=kc,environment=c.environment(),\n   batch=8,length=2048,parameters=34873952,OLD_block_remat=use_remat,\n   existing_OLD_recurrence_remat=True,one_path_one_executable_per_process=True,\n   source_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(__file__).parent.glob(\'*.py\')})\n exe=None\n for seed in (0,1,2):\n  row={\'seed\':seed,\'status\':\'initializing\'};report[\'seeds\'].append(row);save(a,report)\n  host=c.np.random.default_rng(seed).integers(0,256,(8,2048))\n  tokens=c.jnp.asarray(host,c.jnp.int32);params=initialize(cfg,seed,tokens)\n  assert sum(x.size for x in c.jax.tree.leaves(params))==34873952\n  row.update(input_sha256=hashlib.sha256(host.tobytes()).hexdigest(),initial_parameters_sha256=fingerprint(params))\n  state=tx.init(params) if a.phase!=\'gradient\' else None\n  if exe is None:\n   print(\'COMPILE\',a.path,a.phase,\'OLD_block_remat\',use_remat,flush=True)\n   fn=c.jax.value_and_grad(loss) if a.phase==\'gradient\' else step\n   args=(params,tokens) if a.phase==\'gradient\' else (params,state,tokens)\n   exe,seconds=c.compile_fn(fn,args);report[\'compile_s\']=seconds\n   del args  # Do not retain seed-0 params/state while processing later seeds.\n   try:report[\'compiled_memory_analysis\']=str(exe.memory_analysis())\n   except Exception:report[\'compiled_memory_analysis\']=\'unavailable\'\n  row[\'status\']=\'executing\';save(a,report)\n  if a.phase==\'gradient\':\n   result=exe(params,tokens);c.jax.block_until_ready(result)\n   row[\'arrays\']=persist(result,a.intermediate/f\'{a.path}-gradient-seed{seed}.npz\')\n   del result\n  elif a.phase==\'step\':\n   result=exe(params,state,tokens);c.jax.block_until_ready(result)\n   row[\'arrays\']=persist(result,a.intermediate/f\'{a.path}-step-seed{seed}.npz\')\n   # Compute float32 parameter differences on the host, preserving the same gate.\n   delta=c.jax.tree.map(lambda x,p:c.np.asarray(x)-c.np.asarray(p),result[0],params)\n   row[\'deltas\']=persist(delta,a.intermediate/f\'{a.path}-delta-seed{seed}.npz\')\n   del result,delta\n  else:\n   row[\'rounds\']=[]\n   for rnd in range(3):\n    timing=timed_step(exe,params,state,tokens,4 if a.path==\'NEW\' else 2,warmup=2)\n    row[\'rounds\'].append({\'round\':rnd,\'timing\':timing});save(a,report)\n  row[\'status\']=\'completed\';save(a,report);print(\'SEED_COMPLETE\',a.path,a.phase,seed,flush=True)\n  del params,tokens,state;gc.collect()\n report[\'status\']=\'completed\'\n\ndef save(a,report):\n temp=a.output.with_suffix(\'.tmp\');temp.write_text(json.dumps(report,indent=2,allow_nan=False));temp.replace(a.output)\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--path\',choices=(\'NEW\',\'OLD\'),required=True);p.add_argument(\'--phase\',choices=(\'gradient\',\'step\',\'timing\'),required=True)\n p.add_argument(\'--output\',type=Path,required=True);p.add_argument(\'--intermediate\',type=Path,required=True);a=p.parse_args()\n a.intermediate.mkdir(exist_ok=True,parents=True)\n r={\'status\':\'running\',\'path\':a.path,\'phase\':a.phase,\'seeds\':[],\'published\':False}\n save(a,r)\n try:run(a,r)\n except Exception as e:\n  r.update(status=\'error\',error=type(e).__name__,detail=str(e),traceback=traceback.format_exc())\n  print(\'WORKER_ERROR\',a.path,a.phase,type(e).__name__,str(e),flush=True)\n finally:save(a,r)\n print(\'WORKER_FINAL\',a.path,a.phase,r[\'status\'],flush=True)\n raise SystemExit(0 if r[\'status\']==\'completed\' else 1)\n', 'serial_anchor.py': '"""Independent token-serial anchor: three seeds, two chunks, nonzero initial state."""\nimport sys\nimport candidate_common as c\nc.jax.config.update(\'jax_default_matmul_precision\',\'highest\');c.require_tpu()\np,cfg=c.paths(profile=\'no_damping\');rows=[]\nfor seed in (0,1,2):\n args,digest=c.inputs(seed,(1,256,1,128))\n oracle=c.jax.jit(p[\'SERIAL\'])(*args)\n grad=c.jax.jit(c.jax.value_and_grad(c.loss_for(p[\'SERIAL\']),argnums=tuple(range(7))))(*args)\n checks={}\n for name in (\'OLD\',\'JAX_REF\',\'NEW\'):\n  checks[name]={\'forward\':c.check(c.jax.jit(p[name])(*args),oracle),\n   \'all_seven_gradients\':c.check(c.jax.jit(c.jax.value_and_grad(c.loss_for(p[name]),argnums=tuple(range(7))))(*args),grad)}\n passed=all(g[\'passed\'] for v in checks.values() for g in v.values())\n rows.append({\'seed\':seed,\'input_sha256\':digest,\'checks\':checks,\'passed\':passed})\n c.save({\'status\':\'running\',\'environment\':c.environment(),\'config\':cfg,\'seeds\':rows},sys.argv[1])\n print(\'SERIAL_ANCHOR\',seed,passed,flush=True)\nreport={\'status\':\'passed\' if all(r[\'passed\'] for r in rows) else \'gate_failed\',\n \'environment\':c.environment(),\'config\':cfg,\'tolerance_rel_l2\':c.TOL,\'seeds\':rows}\nc.save(report,sys.argv[1]);raise SystemExit(0 if report[\'status\']==\'passed\' else 1)\n', 'all_paths_speed_worker.py': '"""Measure all generations with non-blocking numerical diagnostics. One TPU executable at a time."""\nimport argparse,dataclasses,gc,hashlib,json,statistics,time,traceback\nfrom pathlib import Path\nimport candidate_common as c\nfrom candidate_e2e import initialize,make_functions\nfrom corrected_original_lm import LMConfig\nfrom isolated_e2e import functions,persist,fingerprint\n\ndef select(path):\n paths,cfg=c.paths(profile=\'no_damping\')\n if path==\'PALLAS_V1\':\n  import atomic_ops.gdn2_pipeline as pipeline\n  from atomic_ops import KernelConfig\n  import inspect\n  assert Path(inspect.getsourcefile(pipeline.gdn2_pallas_forward_trainable)).resolve()==(c.ROOT/\'atomic_ops/gdn2_pipeline.py\').resolve()\n  config=KernelConfig(bt=128,bc=64,mb=16,clip=1e4,wy_eps=0.)\n  def fn(q,k,v,w,b,g,h0):return pipeline.gdn2_pallas_forward_trainable(q,k,v,w,b,g,1/128**.5,h0=h0,config=config)\n  return fn,{\'implementation\':\'atomic_ops.gdn2_pipeline.gdn2_pallas_forward_trainable\',\'config\':dataclasses.asdict(config)}\n return paths[path],{\'implementation\':{\'OLD\':\'candidate_common.old_impl / affine associative_scan\',\n  \'JAX_REF\':\'atomic_ops.reference.gdn2_chunked_wy_reference / pure JAX\',\n  \'NEW\':\'atomic_gdn2.pipeline.make_blr_trainable / TPU Pallas\'}[path],\'config\':cfg if path==\'NEW\' else {\'wy_eps\':0.,\'chunk_length\':128,\'matmul_precision\':\'highest\'}}\n\ndef save(a,r):\n temp=a.output.with_suffix(\'.tmp\');temp.write_text(json.dumps(r,indent=2,allow_nan=False));temp.replace(a.output)\n\ndef fixed_timing(exe,args):\n warm=[]\n for _ in range(2):\n  start=time.perf_counter();result=exe(*args);c.jax.block_until_ready(result);warm.append((time.perf_counter()-start)*1000)\n  assert all(c.np.isfinite(c.np.asarray(x)).all() for x in c.jax.tree.leaves(result)),\'Nonfinite warmup result\'\n  del result\n samples=[]\n for _ in range(4):\n  start=time.perf_counter();result=exe(*args);c.jax.block_until_ready(result);samples.append((time.perf_counter()-start)*1000)\n  # Host checks are outside the recorded interval; do not silently time nonfinite computations.\n  assert all(c.np.isfinite(c.np.asarray(x)).all() for x in c.jax.tree.leaves(result)),\'Nonfinite timed result\'\n  del result\n return c.summary(samples)|{\'warmup_samples_ms\':warm,\'fixed_inputs_and_optimizer_state\':True,\n                            \'full_tree_synchronized\':True,\'all_results_finite\':True}\n\ndef run(a,r):\n c.jax.config.update(\'jax_default_matmul_precision\',\'highest\');c.require_tpu()\n fn,identity=select(a.path)\n r.update(path=a.path,kind=a.kind,identity=identity,environment=c.environment(),\n          protocol=\'FIXED_STATE_ALL_PATHS_V1: 2 warmups, 4 timed calls, 3 rounds, 3 seeds; numerical deviations do not block timing\',\n          diagnostic_rel_l2=.005,relative_diagnostic_is_blocking=False,\n          OLD_block_remat=(a.path==\'OLD\' and a.kind==\'e2e\'),\n          scope=\'Raw latency; a numerical diagnostic FAIL is retained and must not be presented as an equivalence PASS\')\n executable=None\n shape=tuple(map(int,a.shape.split(\',\')))\n for seed in (0,1,2):\n  row={\'seed\':seed,\'status\':\'initializing\',\'rounds\':[]};r[\'seeds\'].append(row);save(a,r)\n  if a.kind==\'kernel\':\n   xs,digest=c.inputs(seed,shape);row[\'input_sha256\']=digest;r[\'shape\']=list(shape)\n   # Each seed gets a separate closed-over h0. Remove previous executable/cache first.\n   if executable is not None:del executable;c.jax.clear_caches();gc.collect()\n   def objective(*args):return c.loss_for(fn)(*args,xs[-1])\n   executable,seconds=c.compile_fn(c.jax.grad(objective,argnums=tuple(range(6))),xs[:-1])\n   row[\'compile_seconds\']=seconds\n   observed=executable(*xs[:-1]);c.jax.block_until_ready(observed)\n   row[\'snapshot\']=persist(observed,a.intermediate/f\'{a.label}-seed{seed}.npz\')\n   del observed\n   args=xs[:-1]\n  else:\n   cfg=LMConfig();host=c.np.random.default_rng(seed).integers(0,256,(8,2048))\n   tokens=c.jnp.asarray(host,c.jnp.int32);params=initialize(cfg,seed,tokens)\n   assert sum(x.size for x in c.jax.tree.leaves(params))==34873952\n   loss,step,tx=functions(cfg,fn,a.path==\'OLD\');state=tx.init(params)\n   row.update(input_sha256=hashlib.sha256(host.tobytes()).hexdigest(),initial_parameters_sha256=fingerprint(params),initial_optimizer_sha256=fingerprint(state))\n   initial=a.intermediate/f\'initial-seed{seed}.npz\';receipt=initial.with_suffix(\'.receipt.json\')\n   if not receipt.exists():\n    data=persist(params,initial);data[\'fingerprint\']=row[\'initial_parameters_sha256\'];receipt.write_text(json.dumps(data,indent=2))\n   assert json.loads(receipt.read_text())[\'fingerprint\']==row[\'initial_parameters_sha256\']\n   r.update(parameters=34873952,batch=8,length=2048,model_config=dataclasses.asdict(cfg))\n   if executable is None:\n    executable,seconds=c.compile_fn(step,(params,state,tokens));r[\'compile_seconds\']=seconds\n    try:r[\'compiled_memory\']=str(executable.memory_analysis())\n    except Exception:r[\'compiled_memory\']=\'unavailable\'\n   observed=executable(params,state,tokens);c.jax.block_until_ready(observed)\n   row[\'snapshot\']=persist(observed,a.intermediate/f\'{a.label}-seed{seed}.npz\')\n   delta=c.jax.tree.map(lambda x,p:c.np.asarray(x)-c.np.asarray(p),observed[0],params)\n   row[\'delta\']=persist(delta,a.intermediate/f\'{a.label}-delta-seed{seed}.npz\')\n   del observed,delta\n   args=(params,state,tokens)\n  row[\'status\']=\'timing\';save(a,r)\n  for rnd in range(3):\n   timing=fixed_timing(executable,args);row[\'rounds\'].append({\'round\':rnd,\'timing\':timing});save(a,r)\n  row[\'status\']=\'completed\';save(a,r);print(\'SPEED_SEED\',a.label,seed,[x[\'timing\'][\'median_ms\'] for x in row[\'rounds\']],flush=True)\n  del args\n  if a.kind==\'kernel\':del xs\n  else:del params,state,tokens\n  gc.collect()\n r[\'status\']=\'completed\'\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--path\',choices=(\'OLD\',\'JAX_REF\',\'PALLAS_V1\',\'NEW\'),required=True)\n p.add_argument(\'--kind\',choices=(\'kernel\',\'e2e\'),required=True);p.add_argument(\'--shape\',default=\'4,2048,6,128\')\n p.add_argument(\'--label\',required=True);p.add_argument(\'--output\',type=Path,required=True);p.add_argument(\'--intermediate\',type=Path,required=True);a=p.parse_args()\n r={\'status\':\'running\',\'seeds\':[],\'published\':False}\n try:run(a,r)\n except Exception as e:r.update(status=\'error\',error=type(e).__name__,detail=str(e),traceback=traceback.format_exc());print(\'SPEED_ERROR\',a.label,str(e),flush=True)\n finally:save(a,r)\n print(\'SPEED_FINAL\',a.label,r[\'status\'],flush=True)\n raise SystemExit(0 if r[\'status\']==\'completed\' else 1)\n', 'all_paths_speed_summary.py': '"""Compare fixed-state latencies; retain every numerical deviation as a separate result."""\nimport argparse,csv,json,math,statistics\nfrom pathlib import Path\nimport numpy as np\n\ndef compare(a,b):\n ma=json.loads(a.with_suffix(\'.meta.json\').read_text());mb=json.loads(b.with_suffix(\'.meta.json\').read_text())\n assert ma==mb,\'Different tensor paths/shapes/dtypes\'\n tensors=[]\n with np.load(a,allow_pickle=False) as aa,np.load(b,allow_pickle=False) as bb:\n  for m in ma:\n   av=aa[m[\'key\']].astype(np.float64);bv=bb[m[\'key\']].astype(np.float64)\n   finite=bool(np.isfinite(av).all() and np.isfinite(bv).all())\n   rel=float(np.linalg.norm((av-bv).ravel())/(np.linalg.norm(bv.ravel())+1e-12)) if finite else None\n   tensors.append({\'path\':m[\'path\'],\'finite\':finite,\'rel_l2\':rel,\'passed\':finite and rel<=.005,\n                   \'max_abs\':float(np.max(np.abs(av-bv))) if finite else None})\n return {\'passed\':all(t[\'passed\'] for t in tensors),\'tensors\':tensors}\n\ndef analyze(out,intermediate):\n paths=[\'OLD\',\'JAX_REF\',\'PALLAS_V1\',\'NEW\'];cases=[\'e2e\']+[f\'kernel-h{h}-l{l}\' for h,l in [(6,2048),(2,2048),(2,4096),(4,2048),(4,4096),(6,4096)]]\n rows=[];sample_rows=[];diagnostics=[]\n for case in cases:\n  records={}\n  for path in paths:\n   f=out/f\'{case}-{path}.json\'\n   if not f.exists():continue\n   r=json.loads(f.read_text())\n   if r[\'status\']!=\'completed\':continue\n   assert [s[\'seed\'] for s in r[\'seeds\']]==[0,1,2]\n   for seed in r[\'seeds\']:\n    assert seed[\'status\']==\'completed\' and len(seed[\'rounds\'])==3\n    for rnd in seed[\'rounds\']:\n     t=rnd[\'timing\'];assert len(t[\'samples_ms\'])==4 and len(t[\'warmup_samples_ms\'])==2\n     assert all(math.isfinite(x) and x>0 for x in t[\'samples_ms\']+t[\'warmup_samples_ms\'])\n     assert math.isclose(statistics.median(t[\'samples_ms\']),t[\'median_ms\'],rel_tol=1e-12)\n     for kind,key in [(\'warmup\',\'warmup_samples_ms\'),(\'timed\',\'samples_ms\')]:\n      sample_rows.extend({\'case\':case,\'path\':path,\'seed\':seed[\'seed\'],\'round\':rnd[\'round\'],\'kind\':kind,\'index\':i,\'ms\':x} for i,x in enumerate(t[key]))\n   records[path]=r\n   seed_times=[statistics.median(x[\'timing\'][\'median_ms\'] for x in s[\'rounds\']) for s in r[\'seeds\']]\n   rows.append({\'case\':case,\'path\':path,\'latency_mean_ms\':statistics.mean(seed_times),\'latency_sample_SD_ms\':statistics.stdev(seed_times),\n                \'OLD_over_path_mean\':None,\'OLD_over_path_SD\':None,\'JAX_REF_over_path_mean\':None,\'JAX_REF_over_path_SD\':None})\n  for row in [r for r in rows if r[\'case\']==case]:\n   target=records[row[\'path\']]\n   for baseline in (\'OLD\',\'JAX_REF\'):\n    if baseline not in records:continue\n    reference=records[baseline];ratios=[]\n    for seed in range(3):\n     a=target[\'seeds\'][seed];b=reference[\'seeds\'][seed]\n     assert a[\'input_sha256\']==b[\'input_sha256\']\n     if case==\'e2e\':\n      assert a[\'initial_parameters_sha256\']==b[\'initial_parameters_sha256\']\n      assert a[\'initial_optimizer_sha256\']==b[\'initial_optimizer_sha256\']\n     ratios.append(statistics.median(b[\'rounds\'][i][\'timing\'][\'median_ms\']/a[\'rounds\'][i][\'timing\'][\'median_ms\'] for i in range(3)))\n     # Diag failures are retained and do not invalidate or block raw timing collection.\n     prefix=f\'{case}-{row["path"]}\';refprefix=f\'{case}-{baseline}\'\n     d=compare(intermediate/f\'{prefix}-seed{seed}.npz\',intermediate/f\'{refprefix}-seed{seed}.npz\')\n     item={\'case\':case,\'path\':row[\'path\'],\'reference\':baseline,\'seed\':seed,\'snapshot\':d}\n     if case==\'e2e\':item[\'parameter_difference\']=compare(intermediate/f\'{prefix}-delta-seed{seed}.npz\',intermediate/f\'{refprefix}-delta-seed{seed}.npz\')\n     diagnostics.append(item)\n    row[baseline+\'_over_path_mean\']=statistics.mean(ratios);row[baseline+\'_over_path_SD\']=statistics.stdev(ratios)\n for filename,data in [(\'summary.csv\',rows),(\'all-timing-samples.csv\',sample_rows)]:\n  if data:\n   with (out/filename).open(\'w\',newline=\'\') as f:w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)\n report={\'status\':\'analysis_completed\',\'protocol\':\'FIXED_STATE_ALL_PATHS_V1\',\'speed_is_equivalence_proof\':False,\n         \'note\':\'Non-blocking numerical diagnostics; FAIL stays FAIL. Kernel snapshot is the timed six-input gradient; E2E snapshot is parameters/optimizer/loss, not standalone raw gradients.\',\n         \'rows\':rows,\'diagnostics\':diagnostics}\n (out/\'analysis.json\').write_text(json.dumps(report,indent=2,allow_nan=False))\n for r in rows:print(\'RESULT\',r,flush=True)\n return report\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--results\',type=Path,required=True);p.add_argument(\'--intermediate\',type=Path,required=True);a=p.parse_args();analyze(a.results,a.intermediate)\n'}
PAYLOAD_HASHES={'candidate_common.py': '5b723232a5ddec53890d5830f748b2202f4cda92461abfe6393242a946e4c437', 'candidate_e2e.py': 'db9014e14b7ebefd052a0ca74a680f95b5a4bafdeaaf38afea48f2e170d54bb9', 'corrected_original_lm.py': '8714d76fc9d308f7f1f14585e9a27441a5c7b73ae77217b7b0d90719e24cd1b2', 'isolated_e2e.py': 'a58491fc3bacb70a64d2569458887b679e85a1fd230d7ad6078c878ad2a11016', 'serial_anchor.py': '6ba9b183c28cf900f4d9add9ffd0d70803aac02ad6be2ec401f684448477b0a1', 'all_paths_speed_worker.py': '67d5a2a480880ddb8b85199a8a988c4cfcb841e9d880fea88c1520ec91625884', 'all_paths_speed_summary.py': '6036b174cb46ac3fdb82b4f1d1eb7ef1ef572cdbdd5c717d8556dcda3f239175'}
bench=source/'all_paths_speed';bench.mkdir()
for n,s in PAYLOAD.items():
 p=bench/n;p.write_text(s,encoding='utf-8');assert hashlib.sha256(p.read_bytes()).hexdigest()==PAYLOAD_HASHES[n],n
(OUT/'benchmark-source-manifest.json').write_text(json.dumps(PAYLOAD_HASHES,indent=2))
print('ALL PATH SOURCES VERIFIED')


In [ ]:
import queue,threading,time,statistics,csv,math
import numpy as np
# The notebook parent uses only NumPy. Never import JAX or initialize TPU here.
STATUS={'status':'running','published':False,'cases':[],'protocol':PROTOCOL}
DEST=pathlib.Path('/content')/('Atomic-gdn2-all-paths-speed-results-'+RUN_ID+'.zip')
def collect():
 (OUT/'status.json').write_text(json.dumps(STATUS,indent=2))
 files={'results/'+p.name:p for p in OUT.iterdir() if p.is_file()}
 files.update({'executed-source/'+n:bench/n for n in PAYLOAD})
 files.update({'pinned-source/'+n:source/n for n in EXPECTED_SOURCE_HASHES})
 manifest={'revision':REVISION,'run_id':RUN_ID,'files':{n:hashlib.sha256(p.read_bytes()).hexdigest() for n,p in files.items()}}
 temp=DEST.with_suffix('.tmp')
 with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
  for n,p in files.items():z.write(p,n)
  z.writestr('manifest.json',json.dumps(manifest,indent=2))
 temp.replace(DEST)
 print('CHECKPOINT_ZIP',DEST.name,hashlib.sha256(DEST.read_bytes()).hexdigest(),flush=True)
def execute(label,command):
 print('START',label,flush=True);start=time.monotonic()
 proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 messages=queue.Queue()
 def reader():
  for line in proc.stdout:messages.put(line)
  messages.put(None)
 threading.Thread(target=reader,daemon=True).start()
 try:
  with (OUT/(label+'.log')).open('w') as sink:
   while True:
    try:line=messages.get(timeout=30)
    except queue.Empty:
     print('PROGRESS',label,int(time.monotonic()-start),'seconds',flush=True)
     if time.monotonic()-start>5400:
      proc.terminate()
      try:proc.wait(timeout=15)
      except subprocess.TimeoutExpired:proc.kill()
     continue
    if line is None:break
    sink.write(line);sink.flush();print(line,end='',flush=True)
   code=proc.wait()
 finally:
  if proc.poll() is None:
   proc.terminate()
   try:proc.wait(timeout=15)
   except subprocess.TimeoutExpired:proc.kill();proc.wait()
 path=OUT/(label+'.json');result=json.loads(path.read_text()) if path.exists() else {'status':'missing_result'}
 STATUS['cases'].append({'case':label,'status':result['status'],'exit_code':code,'elapsed_s':time.monotonic()-start});collect()
 return result

try:
 # Preserve the independent diagnostic, but even its numerical FAIL does not stop raw timers.
 execute('serial-anchor',[PY,'-u',str(bench/'serial_anchor.py'),str(OUT/'serial-anchor.json')])
 cases=[('e2e','e2e',None)]+[(f'kernel-h{h}-l{l}','kernel',f'4,{l},{h},128') for h,l in [(6,2048),(2,2048),(2,4096),(4,2048),(4,4096),(6,4096)]]
 paths=['NEW','OLD','JAX_REF','PALLAS_V1']
 for index,(case,kind,shape) in enumerate(cases):
  order=paths[index%len(paths):]+paths[:index%len(paths)]
  for path in order:
   label=case+'-'+path
   command=[PY,'-u',str(bench/'all_paths_speed_worker.py'),'--path',path,'--kind',kind,'--label',label,
     '--output',str(OUT/(label+'.json')),'--intermediate',str(INTERMEDIATE)]
   if shape is not None:command+=['--shape',shape]
   execute(label,command)
 STATUS['status']='timing_completed'
 STATUS['runtime_error_cases']=[r['case'] for r in STATUS['cases'] if r['exit_code']!=0]
except KeyboardInterrupt:STATUS['status']='interrupted'
except Exception as e:STATUS.update(status='error',detail=str(e));print('SUITE_ERROR',str(e),flush=True)
finally:collect()
print('TIMING_COLLECTION_FINISHED',json.dumps(STATUS),flush=True)


In [ ]:
# Analyze diagnostics after timing collection; diagnostics never block remaining speed cases.
analysis=execute('analysis',[PY,'-u',str(bench/'all_paths_speed_summary.py'),'--results',str(OUT),'--intermediate',str(INTERMEDIATE)])
p=OUT/'analysis.json'
if p.exists():
 r=json.loads(p.read_text())
 for row in r['rows']:print(row)
 print('NUMERICAL_DIAGNOSTIC_FAILURES',sum(not d['snapshot']['passed'] or not d.get('parameter_difference',{'passed':True})['passed'] for d in r['diagnostics']))
 print('These failures remain failures. The times are raw latency measurements, not equivalence claims.')
collect()
with zipfile.ZipFile(DEST) as z:
 m=json.loads(z.read('manifest.json'))
 assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in m['files'].items())
print('ZIP',DEST,'SHA256',hashlib.sha256(DEST.read_bytes()).hexdigest())
print('Large numerical snapshots remain in',INTERMEDIATE,'; compact diagnostics and sample-level timings are in ZIP.')
from google.colab import files
files.download(str(DEST))
